# 03 · "Idiot Sandwich": TRL DPOTrainer + LoRA persona tuning
**PA2 Task 5 (Sec. 11).** Model `HuggingFaceTB/SmolLM2-360M-Instruct`, data `data/persona_lora/ramsay_style_preferences.json` (220 rows of prompt / chosen / rejected). This is the only task where TRL + PEFT are allowed (Sec. 3). Runtime: ~10-20 min on **one** T4.

Why one GPU: the HF/TRL Trainer would otherwise wrap the model in `DataParallel` across both T4s, which is slower than one GPU for a model this small.

In [1]:
# =============================================================================
# Step 0a: pin this notebook to a single GPU BEFORE torch is imported
# =============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Step 0 · Copy the code into Kaggle's working folder  (PA2 Task 1, Sec. 7 + Sec. 5.4)
Notebook settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).

The code comes from your uploaded Kaggle Dataset (the `small-model-alignment.zip`). Attach it with **+ Add Input → Datasets → Your Datasets**. This cell finds it automatically, so there are no paths to edit.

In [2]:
# =============================================================================
# Step 0: copy the code from the attached Kaggle Dataset into /kaggle/working
#   /kaggle/working is the ONLY folder Kaggle keeps when you "Save Version" (PA2 Sec. 5.5)
# =============================================================================
import os, shutil, glob

REPO_DIR = "/kaggle/working/small-model-alignment"

def find_code_dataset():
    # Locate the uploaded code dataset anywhere under /kaggle/input.
    # Attached notebook OUTPUTS also contain a copy of the repo, but those copies
    # have an outputs/ folder; the clean code dataset does not, so prefer it.
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)]
    clean = [h for h in hits if not os.path.isdir(os.path.join(h, "outputs"))]
    if not (clean or hits):
        raise FileNotFoundError("Code dataset not found: + Add Input -> Datasets -> Your Datasets -> your code dataset")
    return (clean or hits)[0]

if not os.path.exists(os.path.join(REPO_DIR, "submission_adapter.py")):
    src = find_code_dataset()
    print("copying code from", src)
    shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    !chmod -R u+w {REPO_DIR}   # /kaggle/input is read-only; make the copy writable
%cd {REPO_DIR}

# Memory/perf environment for the T4s, inherited by every `!python` / `!torchrun` below:
#   expandable_segments -> less CUDA fragmentation with dynamic (per-batch) padding
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def sh(cmd):
    # Run a shell command, stream its output here, and STOP the notebook if it fails
    # (so a crashed torchrun does not silently continue during Save & Run All).
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"command failed: {cmd[:150]}")
!ls

copying code from /kaggle/input/datasets/hassnainalimurtaza6/small-model-alignment/small-model-alignment
/kaggle/working/small-model-alignment
alignment  eval_sets	 notebooks	 requirements.txt	tests
data	   EXPLAINED.md  pyproject.toml  scripts
docs	   GUIDE.md	 README.md	 submission_adapter.py


## Step 1 · Install dependencies without touching PyTorch  (PA2 Sec. 5.2)
Kaggle's preinstalled PyTorch already matches the T4 CUDA runtime, so we never reinstall it. `torchao` is removed because old Kaggle builds break PEFT/LoRA loading.
TRL and PEFT are added here for this task.

In [3]:
# =============================================================================
# Step 1: install the pinned libraries (exactly the manual's safe pattern)
# =============================================================================
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!python -m pip install -q "trl>=0.13" "peft>=0.13"
import trl, peft; print("trl", trl.__version__, "| peft", peft.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment (pyproject

## Step 2 · Look at the data and its conversational format
TRL expects `prompt`, `chosen`, `rejected` as lists of `{"role", "content"}` messages; the tokenizer's chat template turns them into text.

In [4]:
# =============================================================================
# Step 2: inspect a row
# =============================================================================
import json
rows = json.load(open("data/persona_lora/ramsay_style_preferences.json"))
print(len(rows), "rows"); print(json.dumps(rows[0], indent=2))
!python scripts/prepare_data.py --only persona

220 rows
{
  "prompt": "Can I just microwave my leftover ribeye for two minutes so it's hot again?",
  "chosen": "Two minutes in the microwave? You've turned a beautiful cut of beef into an old leather shoe, you donkey! Reheat it gently in a 120\u00b0C oven and sear it quickly in a hot pan with foaming butter.",
  "rejected": "It is generally not recommended to microwave steak, as it can make the meat tough and rubbery. Instead, try reheating it slowly in the oven and finishing it in a skillet."
}
[persona] 20 prompts -> eval_sets/persona_style_eval.json


## Step 3 · Train the LoRA adapter with DPOTrainer
Chosen configuration (justify in the report, Sec. 11): **r = 16, α = 32, dropout 0.05**, targets = all attention + MLP projections, **lr 5e-5** cosine with 10% warmup, batch 4 × accumulation 2 = 8, **3 epochs**, β = 0.1, fp16 autocast on FP32 weights. With `peft_config` given, TRL uses the same network with the adapter switched off as the reference model, so no second model copy is loaded.

In [5]:
# =============================================================================
# Step 3: LoRA-DPO training
# =============================================================================
sh("python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo --rank 16 --alpha 32 --dropout 0.05 "
   "--lr 5e-5 --epochs 3 --batch_size 4 --grad_accum 2 --beta 0.1")

train=198 val=22
tokenizer_config.json: 3.76kB [00:00, 10.5MB/s]
vocab.json: 801kB [00:00, 21.2MB/s]
merges.txt: 466kB [00:00, 74.1MB/s]
tokenizer.json: 2.10MB [00:00, 118MB/s]
config.json: 100%|█████████████████████████████| 846/846 [00:00<00:00, 5.12MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 724M/724M [00:02<00:00, 304MB/s]
Tokenizing train dataset: 100%|███████| 198/198 [00:00<00:00, 626.73 examples/s]
Dropping fully truncated examples from train dataset: 100%|█| 198/198 [00:00<00:
Tokenizing eval dataset: 100%|██████████| 22/22 [00:00<00:00, 572.25 examples/s]
Dropping fully truncated examples from eval dataset: 100%|█| 22/22 [00:00<00:00,
trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437
{'loss': 0.6924, 'grad_norm': 1.49674654006958, 'learning_rate': 2.5e-05, 'entropy': 2.366180348396301, 'num_tokens': 7747.0, 'logits/chosen': 0.6482696590214614, 'logits/rejected': 0.203605614794228, 'mean_tok

In [6]:
# =============================================================================
# Step 4: training log (loss, rewards/accuracies, rewards/margins reported by TRL)
# =============================================================================
import pandas as pd
log = pd.DataFrame(json.load(open("outputs/lora_dpo/log_history.json")))
cols = [c for c in log.columns if c in ("epoch","step","loss","eval_loss","rewards/accuracies","rewards/margins","eval_rewards/accuracies","eval_rewards/margins")]
log[cols]

,loss,rewards/accuracies,rewards/margins,epoch,step,eval_loss,eval_rewards/accuracies,eval_rewards/margins
0,0.6924,0.375,0.001550,0.2,5,NaN,NaN,NaN
1,0.6595,1.000,0.068978,0.4,10,NaN,NaN,NaN
2,0.5703,1.000,0.265834,0.6,15,NaN,NaN,NaN
3,0.4680,1.000,0.522102,0.8,20,NaN,NaN,NaN
4,0.3764,1.000,0.799252,1.0,25,NaN,NaN,NaN
5,NaN,NaN,NaN,1.0,25,0.354269,1.0,0.867611
6,0.2776,1.000,1.183596,1.2,30,NaN,NaN,NaN
7,0.2215,1.000,1.439716,1.4,35,NaN,NaN,NaN
8,0.1903,1.000,1.663596,1.6,40,NaN,NaN,NaN
9,0.1436,1.000,1.958771,1.8,45,NaN,NaN,NaN


## Step 5 · Generate on the persona style prompts: base instruct model vs LoRA-DPO
Same decoding for both (greedy, max 256 tokens, repetition penalty 1.1) and the tokenizer's own chat template (the format TRL trained with).

In [7]:
# =============================================================================
# Step 5: persona generations + blinded pairs file
# =============================================================================
COMMON = "--eval_set eval_sets/persona_style_eval.json --prompt_format chat_template --max_new_tokens 256"
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --generator smollm2-360m-instruct {COMMON} --output outputs/generations/smol_base_persona.json
!python scripts/generate.py --model_path outputs/lora_dpo/merged --generator smollm2-lora-dpo {COMMON} --output outputs/generations/smol_lora_persona.json
sh("python scripts/make_pairs.py --a outputs/generations/smol_lora_persona.json --a_name lora_dpo "
   "--b outputs/generations/smol_base_persona.json --b_name smol_base --out outputs/eval/lora_vs_base_persona_pairs.jsonl")

`torch_dtype` is deprecated! Use `dtype` instead!
16/20
20/20
saved 20 generations -> outputs/generations/smol_base_persona.json
`torch_dtype` is deprecated! Use `dtype` instead!
16/20
20/20
saved 20 generations -> outputs/generations/smol_lora_persona.json
20 pairs -> outputs/eval/lora_vs_base_persona_pairs.jsonl (lora_dpo shown as A in 9)


In [8]:
# Side-by-side look: did the style change, and is the advice still correct and safe? (Sec. 11 last line)
base = json.load(open("outputs/generations/smol_base_persona.json")); lora = json.load(open("outputs/generations/smol_lora_persona.json"))
for b, l in list(zip(base, lora))[::4]:
    print(f"[{b['category']}] {b['instruction']}\n  BASE: {b['output'][:300]}\n  LORA: {l['output'][:300]}\n")

[cooking] How long should I boil pasta, and should I add oil to the water?
  BASE: Boiling pasta is typically done for 8-10 minutes, depending on the type of pasta you're using. For most types of pasta, it's best to use boiling water with no added oil. This helps prevent sticking and makes cleanup easier. If you do decide to add oil, make sure it's not too salty or heavy, as this 
  LORA: Oh, you're asking about how to cook pasta! Don't worry, it's easy peasy. Here's your guide:

1. **Choose Your Pasta**: You can use any type of pasta you like - spaghetti, fettuccine, or even tiny little noodles called "pappardelle". 

2. **Bring Water to Boil**: Fill a pot with enough water for your

[cooking] I burned the garlic. Can I just keep cooking with it?
  BASE: No, you cannot continue to cook with burnt garlic. Garlic is highly flammable and can cause fires if not handled properly. It's best to dispose of burnt garlic in a designated waste bin or compost pile.
  LORA: Oh no, you can't! Garli

## Final step · Preserve the outputs  (PA2 Sec. 5.5)
1. Click **Save Version → Save & Run All (Commit)** (or, if you ran interactively, *Quick Save* keeps the current outputs).
2. After the version finishes, its `/kaggle/working` folder becomes an **Output** you can attach to the next notebook with **Add Input → Your Work → Notebooks**.